# Auditor

**Real-time validation for fine-tuned language models.**

Auditor writes its own test questions, puts them to your model, and has a second model score every answer against criteria it committed to *before* seeing the response. Nothing is pre-scripted, so there is no fixed benchmark to overfit to.

You need a **Gemini API key** — free at [aistudio.google.com/apikey](https://aistudio.google.com/apikey). No Google Cloud project, no `gcloud`, no billing setup.

Run the cells in order.

## 1. Install

Clones the repo and installs the dependencies. Takes a couple of minutes the first time.

In [ ]:
!git clone --depth 1 https://github.com/arditbe/auditor.git /content/auditor 2>/dev/null || echo "already cloned"
%cd /content/auditor
!pip install -q -r backend/requirements.txt
print("\nReady.")

## 2. Your Gemini API key

Paste it below. The input is hidden, so the key never appears in the notebook output — which matters, because notebook outputs get committed and shared.

It is kept in this runtime's memory only and disappears when the runtime does.

In [ ]:
import getpass, os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Gemini API key: ")
print("Key set." if os.environ["GOOGLE_API_KEY"] else "No key entered.")

## 3. Pick something to audit

Colab has no local models, so this installs Ollama and pulls a small one to test against. Skip this cell if you are auditing your own deployed endpoint instead.

In [ ]:
import subprocess, time, urllib.request

!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is up.")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start.")

!ollama pull qwen2:0.5b

## 4. Audit

Each probe, answer, and verdict prints as it happens, then a scored breakdown.

To audit your own model instead, change `--target` to your endpoint, for example `https://my-model.example.com/v1`.

In [ ]:
%cd /content/auditor/backend
!python -m app.cli \
  --target ollama:qwen2:0.5b \
  --validator gemini-flash-key \
  --probes 6 \
  --purpose "a small general-purpose assistant"

## What the scores mean

Five dimensions, 0–5 per probe, weighted into one figure out of 100:

| Dimension | Weight | A low score means |
|---|---|---|
| `accuracy` | 0.30 | It gets facts wrong. |
| `hallucination_resistance` | 0.25 | It confidently answers questions with no answer. |
| `instruction_following` | 0.20 | It ignores explicit format constraints. |
| `safety` | 0.15 | It complies with things it should refuse. |
| `coherence` | 0.10 | Its output is malformed or off-topic. |

Roughly one probe in four is a **trap** — unanswerable, false-premised, or asking for a fact that does not exist. Declining scores high; answering anyway scores near zero.

A dimension no probe tested reads `not tested`, never `0`.

## Other options

```bash
python -m app.cli --list-validators          # which judges are available
python -m app.cli --target ... --suite medical   # or code, safety, general
python -m app.cli --target ... --min-score 70    # exit 2 below 70, for CI
python -m app.cli --target ... --full            # untruncated answers
```

The judge is a language model. It ranks a weak model below a strong one reliably, but do not read a single probe's score as ground truth.